In [ ]:
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import dask
import numpy as np
import plotly.graph_objects as go
import importlib
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import src.utils.stochastic as stochastic

importlib.reload(stochastic)

load_dotenv()

# Infrastructure parameters
POSTGRES_URL = os.getenv("POSTGRES_URL")
CLUSTER_TYPE = "local"
N_WORKERS = 6
N_CONCCURENT_DATABASE_CALLS = 10

# Model parameters
DISCOUNT_RATE = 0.001  # Discount rate
TRANSACTION_COST = 0.001  # Transaction cost
PVALUE_THRESHOLD = 0.001  # Only trade if we have 99.9% confidence
PERCENT_LOSS = 0.05
CASH_ALLOCATION = 1000

dask.config.set({"distributed.scheduler.locks.lease-timeout": "120s"})  # 2 minutes

engine = create_engine(POSTGRES_URL)

In [ ]:
def f_exit_level(x, mu, sigma, theta, r, c, use_analytical=True):
    return (x - c) * stochastic.OrnsteinUhlenbeck.F(
        x,
        mu,
        sigma,
        theta=theta,
        r=r,
        derivative=1,
        use_analytical=use_analytical,
    ) - stochastic.OrnsteinUhlenbeck.F(
        x, mu, sigma, theta=theta, r=r, use_analytical=use_analytical
    )


def f_prime_exit_level(
    x,
    mu,
    sigma,
    theta,
    r,
    c,
    use_analytical=True,
):
    return (x - c) * stochastic.OrnsteinUhlenbeck.F(
        x,
        mu,
        sigma,
        theta=theta,
        r=r,
        derivative=2,
        use_analytical=use_analytical,
    )


# Function f(x) operating on spreads
def f_entry_level(x, mu, sigma, theta, r, c, exit_level, use_analytical=True):
    return (
        stochastic.OrnsteinUhlenbeck.G(
            x, mu, sigma, theta=theta, r=r, derivative=1, use_analytical=use_analytical
        )
        * (
            stochastic.OrnsteinUhlenbeck.V(
                x,
                mu,
                sigma,
                theta=theta,
                r=r,
                c=c,
                exit_level=exit_level,
                use_analytical=use_analytical,
            )
            - x
            - c
        )
    ) - (
        stochastic.OrnsteinUhlenbeck.G(
            x, mu, sigma, theta=theta, r=r, use_analytical=use_analytical
        )
        * (
            stochastic.OrnsteinUhlenbeck.V_prime(
                x,
                mu,
                sigma,
                theta=theta,
                r=r,
                c=c,
                exit_level=exit_level,
                use_analytical=use_analytical,
            )
            - 1
        )
    )


# Derivative of f(x) via finite difference
def f_prime_entry_level(x, mu, sigma, theta, r, c, exit_level, use_analytical=True):
    return (
        stochastic.OrnsteinUhlenbeck.G(
            x, mu, sigma, theta=theta, r=r, derivative=2, use_analytical=use_analytical
        )
        * (
            stochastic.OrnsteinUhlenbeck.V(
                x,
                mu,
                sigma,
                theta=theta,
                r=r,
                c=c,
                exit_level=exit_level,
                use_analytical=use_analytical,
            )
            - x
            - c
        )
    ) - (
        stochastic.OrnsteinUhlenbeck.G(
            x, mu, sigma, theta=theta, r=r, use_analytical=use_analytical
        )
        * (
            stochastic.OrnsteinUhlenbeck.V_double_prime(
                x,
                mu,
                sigma,
                theta=theta,
                r=r,
                c=c,
                exit_level=exit_level,
                use_analytical=use_analytical,
            )
        )
    )

In [ ]:
# def f_exit_level(x, mu, sigma, theta, r, c, use_analytical=True):
#     return (x - c) * stochastic.OrnsteinUhlenbeck.F(
#         x,
#         mu,
#         sigma,
#         theta=theta,
#         r=r,
#         derivative=1,
#         use_analytical=use_analytical,
#     ) - stochastic.OrnsteinUhlenbeck.F(
#         x, mu, sigma, theta=theta, r=r, use_analytical=use_analytical
#     )


# def f_prime_exit_level(
#     x,
#     mu,
#     sigma,
#     theta,
#     r,
#     c,
#     h=1e-6,
#     use_analytical=True,
# ):
#     return (
#         f_exit_level(x + h, mu, sigma, theta, r=r, c=c, use_analytical=use_analytical)
#         - f_exit_level(x - h, mu, sigma, theta, r=r, c=c, use_analytical=use_analytical)
#     ) / (2 * h)


# # Function f(x) operating on spreads
# def f_entry_level(x, mu, sigma, theta, r, c, exit_level, use_analytical=True):
#     return stochastic.OrnsteinUhlenbeck.G(
#         x, mu, sigma, theta=theta, r=r, use_analytical=use_analytical
#     ) * (
#         stochastic.OrnsteinUhlenbeck.V_prime(
#             x,
#             mu,
#             sigma,
#             theta=theta,
#             r=r,
#             c=c,
#             exit_level=exit_level,
#             use_analytical=use_analytical,
#         )
#         - 1
#     ) - stochastic.OrnsteinUhlenbeck.G(
#         x, mu, sigma, theta=theta, r=r, derivative=1, use_analytical=use_analytical
#     ) * (
#         stochastic.OrnsteinUhlenbeck.V(
#             x,
#             mu,
#             sigma,
#             theta=theta,
#             r=r,
#             c=c,
#             exit_level=exit_level,
#             use_analytical=use_analytical,
#         )
#         - x
#         - c
#     )


# # Derivative of f(x) via finite difference
# def f_prime_entry_level(x, mu, sigma, theta, r, c, exit_level, h=1e-6, use_analytical=True):
#     return (
#         f_entry_level(x + h, mu, sigma, theta, r, c, exit_level, use_analytical=use_analytical)
#         - f_entry_level(x - h, mu, sigma, theta, r, c, exit_level, use_analytical=use_analytical)
#     ) / (2 * h)

In [ ]:
r = DISCOUNT_RATE
c = TRANSACTION_COST

In [ ]:
mu = 0.036945476519220912
sigma = 0.001
theta = 0.0
r = 0.01
c = 0.01
exit_level = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(mu, sigma, theta, r, c)
exit_level

In [ ]:
entry_level = stochastic.OrnsteinUhlenbeck.get_optimal_entry_level(
    mu, sigma, theta, exit_level, r, c
)
entry_level

In [ ]:
mu = 0.05059847509762849
sigma = 0.1
theta = 0.0
r = 0.01
c = 0.01
N = 1000
max_iterations = 1000
tol = 1e-6
b_linspace = np.linspace(theta, theta + 100 * sigma, N)
f_grid_analytical = f_exit_level(
    b_linspace,
    mu=mu,
    sigma=sigma,
    theta=theta,
    r=r,
    c=c,
)
f_grid = f_exit_level(
    b_linspace, mu=mu, sigma=sigma, theta=theta, r=r, c=c, use_analytical=True
)
f_prime_grid = f_prime_exit_level(
    b_linspace, mu=mu, sigma=sigma, theta=theta, r=r, c=c, use_analytical=True
)
f_search = np.where((f_grid > 0) & (f_prime_grid > 0), f_grid, np.inf)
b_x0 = b_linspace[np.argmin(f_search)]
tol = 1e-6
b = b_x0
for i in range(max_iterations):
    print(f"Iteration {i}: b = {b}, f(b) = {f_exit_level(b, mu, sigma, theta, r, c)}")
    f_val = f_exit_level(b, mu, sigma, theta, r, c)
    fp_val = f_prime_exit_level(b, mu, sigma, theta, r, c)
    if fp_val == 0 or not np.isfinite(fp_val) or not np.isfinite(f_val):
        break
    b = b - f_val / fp_val
    if np.abs(f_val) < tol:
        break
b = b[0]
fig = go.Figure()
x = np.linspace(b * 0.9, b * 1.1, N)
y = f_exit_level(x, mu=mu, sigma=sigma, theta=theta, r=r, c=c, use_analytical=True)
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="f(b) near optimum"))
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="")
fig.add_vline(
    x=b,
    line_dash="dash",
    line_color="green",
    annotation_text=f"Optimal exit level: {b}",
)
fig.update_layout(
    title="f(b) vs b",
    xaxis_title="b",
    yaxis_title="f(b)",
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True),
    showlegend=True,
)
fig.show()

In [ ]:
N = 1000
max_iterations = 1000
tol = 1e-6
print(f"Exit level spread: {b}")
d_linspace = np.linspace(theta - 100 * sigma, theta, N)
f_grid = f_entry_level(
    d_linspace,
    mu=mu,
    sigma=sigma,
    theta=theta,
    r=r,
    c=c,
    exit_level=b,
    use_analytical=True,
)
f_prime_grid = f_prime_entry_level(
    d_linspace,
    mu=mu,
    sigma=sigma,
    theta=theta,
    r=r,
    c=c,
    exit_level=b,
    use_analytical=True,
)
f_search = np.where((f_grid < 0) & (f_prime_grid > 0), f_grid, np.inf)
d_x0 = d_linspace[np.argmin(f_search)]
d = d_x0
for i in range(max_iterations):
    print(
        f"Iteration {i}: d = {d}, f(d) = {f_entry_level(d, mu, sigma, theta, r, c, b, use_analytical=True)}"
    )
    f_val = f_entry_level(d, mu, sigma, theta, r, c, b, use_analytical=True)
    fp_val = f_prime_entry_level(d, mu, sigma, theta, r, c, b, use_analytical=True)
    if fp_val == 0 or not np.isfinite(fp_val) or not np.isfinite(f_val):
        break
    d = d - f_val / fp_val
    if np.abs(f_val) < tol:
        break
d = d[0]
fig = go.Figure()
x = np.linspace(d * 0.9, d * 1.1, N)
y = f_entry_level(
    x, mu=mu, sigma=sigma, theta=theta, r=r, c=c, exit_level=b, use_analytical=True
)
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="f(d)"))
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="")
fig.add_vline(
    x=d,
    line_dash="dash",
    line_color="green",
    annotation_text=f"Optimal entry level: {d}",
)
fig.update_layout(
    title="f(d) vs d",
    xaxis_title="d",
    yaxis_title="f(d)",
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True),
    showlegend=True,
)
fig.show()

In [ ]:
import importlib

importlib.reload(stochastic)

mu = 0.05059847509762849
sigma = 0.1
theta = 0.0
r = 0.01
c = 0.01

# Get exit level
exit_level_arr = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(
    np.array([mu]), np.array([sigma]), np.array([theta]), r, c
)
print(f"Exit level: {exit_level_arr[0]}")

# Test the grid search for entry level manually
from src.utils.stochastic._non_rolling import OrnsteinUhlenbeck

theta_v = np.array([theta])
sigma_v = np.array([sigma])
mu_v = np.array([mu])
exit_level_v = exit_level_arr

# Create grid
x_grid = np.linspace(theta_v - 100 * sigma_v, theta_v, 1000)
print(f"Grid shape: {x_grid.shape}")
print(f"Grid range: {x_grid.min()} to {x_grid.max()}")


# Test f_entry_level on grid
def test_f_entry(x, mu, sigma, theta, r, c, exit_level):
    return (
        OrnsteinUhlenbeck.G(
            x, mu, sigma, theta=theta, r=r, derivative=1, use_analytical=True
        )
        * (
            OrnsteinUhlenbeck.V(
                x,
                mu,
                sigma,
                theta=theta,
                r=r,
                c=c,
                exit_level=exit_level,
                use_analytical=True,
            )
            - x
            - c
        )
    ) - (
        OrnsteinUhlenbeck.G(x, mu, sigma, theta=theta, r=r, use_analytical=True)
        * (
            OrnsteinUhlenbeck.V_prime(
                x,
                mu,
                sigma,
                theta=theta,
                r=r,
                c=c,
                exit_level=exit_level,
                use_analytical=True,
            )
            - 1
        )
    )


f_grid = test_f_entry(x_grid, mu_v, sigma_v, theta_v, r, c, exit_level_v)
print(f"f_grid shape: {f_grid.shape}")
print(f"f_grid min/max: {np.nanmin(f_grid)} / {np.nanmax(f_grid)}")
print(f"Number of negative values: {np.sum(f_grid < 0)}")